In [1]:
from pathlib import Path
import subprocess
import numpy as np 
import random as rand
import pandas as pd
import time
from typing import List, Dict
import json

import time
import requests
from Bio.Align import substitution_matrices
from Bio import Entrez
from Bio import SeqIO
import matplotlib.pyplot as plt
import logomaker
import random
from io import StringIO

# **1. Fetching and sampling SwissProt Mouse Proteome**

In [9]:
BASE_URL    = "https://rest.uniprot.org/uniprotkb"
PROTEOME_ID = "UP000000589"   # Mus musculus reference proteome
QUERY       = f"proteome:{PROTEOME_ID} AND reviewed:true"
 
OUT_DIR  = Path("/home/aalarkin/Work/anchorminer/datasets/RunningMHCPan")
PEP_OUT  = OUT_DIR / "peptide_samples_mouse_mhcI.txt"
 
N_PEPTIDES = 1_000_000
MIN_LEN, MAX_LEN = 8, 12
SEED = 42
 
 
def fetch_fasta(query: str) -> str:
    session = requests.Session()
    session.headers["User-Agent"] = "Python/UniProt-MouseProteome-Downloader"
    params = {"query": query, "format": "fasta", "includeIsoform": "true", "compressed": "false"}
    print("Streaming Swiss-Prot mouse FASTA from UniProt...")
    with session.get(f"{BASE_URL}/stream", params=params, stream=True, timeout=300) as r:
        r.raise_for_status()
        chunks = []
        for chunk in r.iter_content(chunk_size=1 << 20, decode_unicode=True):
            if chunk:
                chunks.append(chunk)
    return "".join(chunks)
 
 
def sample_peptides(sequences, n, min_len, max_len, rng):
    eligible = [s for s in sequences if len(s) >= min_len]
    peptides = []
    while len(peptides) < n:
        protein = rng.choice(eligible)
        pep_len = rng.randint(min_len, min(max_len, len(protein)))
        start   = rng.randint(0, len(protein) - pep_len)
        peptides.append(str(protein[start: start + pep_len]))
    return peptides
 
 
fasta_text = fetch_fasta(QUERY)
FASTA_OUT = OUT_DIR / "mouse_swissprot.fasta"
FASTA_OUT.write_text(fasta_text)

sequences  = [r.seq for r in SeqIO.parse(StringIO(fasta_text), "fasta")]
print(f"{len(sequences):,} proteins loaded")
 
rng      = random.Random(SEED)
print(f"Sampling {N_PEPTIDES:,} peptides (len {MIN_LEN}–{MAX_LEN})...")
peptides = sample_peptides(sequences, N_PEPTIDES, MIN_LEN, MAX_LEN, rng)
 
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(PEP_OUT, "w") as fh:
    fh.write("\n".join(peptides) + "\n")
 
print(f"Done → {PEP_OUT}  ({len(peptides):,} peptides)")


Streaming Swiss-Prot mouse FASTA from UniProt...
25,709 proteins loaded
Sampling 1,000,000 peptides (len 8–12)...
Done → /home/aalarkin/Work/anchorminer/datasets/RunningMHCPan/peptide_samples_mouse_mhcI.txt  (1,000,000 peptides)


# **2. Generating background aminoacid frequencies**

In [17]:
concated_seqs = ''
for record in SeqIO.parse('../datasets/mouse_proteome/mouse_swissprot.fasta', "fasta"):
    concated_seqs += str(record.seq)


alphabet = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

_len_proteome = len(concated_seqs)
print('Total len of proteome:', _len_proteome, 'aminoacids')
background_freq = {alphabet[i]: (concated_seqs.count(alphabet[i])/_len_proteome)      for i in range(len(alphabet))}

print('Aminoacids frequency in mouse proteome (background frequency):', _len_proteome) 

for i in background_freq:
    print(i, background_freq[i])

print('Total', sum(list(background_freq.values())))


Total len of proteome: 15170599 aminoacids
Aminoacids frequency in mouse proteome (background frequency): 15170599
A 0.06956053613967385
R 0.05651147986971378
N 0.03560841598937524
D 0.049105509940642424
C 0.021596642294743933
Q 0.04854159021670799
E 0.07167706430049334
G 0.06439172243627295
H 0.02554078451351855
I 0.04277194328318875
L 0.09971616809593346
K 0.05685846682784246
M 0.021548391068803546
F 0.0357153333233579
P 0.06267933125119186
S 0.08533769826755028
T 0.053980729435930644
W 0.01182029793286343
Y 0.02626969442669996
V 0.060760422182406904
Total 0.9999922217969113


# **3. Checkig number of strong binders per MHC**

In [19]:
PATH_TO_NETMHCPAN_RES = '/home/aalarkin/Work/anchorminer/datasets/RunningMHCPan/Data_Per_MHC_Mouse'
BINDING_THRESHOLD = 0.75
N_MIN_BINDERS = 200

files  = subprocess.run(['ls', PATH_TO_NETMHCPAN_RES], 
                       capture_output=True, 
                       text=True).stdout.split('\n')


files = files[:-1]  #last list elem is ''
lenghts = [8,9,10,11,12]

PWM_data = dict()

for file in files:

    df = pd.read_csv(f'{PATH_TO_NETMHCPAN_RES}/{file}', sep = '\t', skiprows = 1)
    df['lenght'] = df['Peptide'].str.len()
    df = df[df['Rank'] < BINDING_THRESHOLD]
    

    label = file.split('.')[0]
    print(label)

    for l in lenghts:
        n_binders = len(df[df['lenght'] == l])
        print(f'len: {l}, number of binders: {n_binders}, binding threshold {BINDING_THRESHOLD}')
        
        if n_binders > N_MIN_BINDERS:

            peptides_merged = ','.join(df[df['lenght'] == l ]['Peptide'].tolist())

            if label not in PWM_data:
                PWM_data[label] = dict()

            PWM_data[label][l] = peptides_merged
    

    print('=' * 50)





H_2_Db
len: 8, number of binders: 175, binding threshold 0.75
len: 9, number of binders: 3276, binding threshold 0.75
len: 10, number of binders: 1086, binding threshold 0.75
len: 11, number of binders: 1007, binding threshold 0.75
len: 12, number of binders: 160, binding threshold 0.75
H_2_Dd
len: 8, number of binders: 1584, binding threshold 0.75
len: 9, number of binders: 3549, binding threshold 0.75
len: 10, number of binders: 877, binding threshold 0.75
len: 11, number of binders: 766, binding threshold 0.75
len: 12, number of binders: 158, binding threshold 0.75
H_2_Dq
len: 8, number of binders: 652, binding threshold 0.75
len: 9, number of binders: 2800, binding threshold 0.75
len: 10, number of binders: 1759, binding threshold 0.75
len: 11, number of binders: 1520, binding threshold 0.75
len: 12, number of binders: 228, binding threshold 0.75
H_2_Kb
len: 8, number of binders: 3152, binding threshold 0.75
len: 9, number of binders: 3062, binding threshold 0.75
len: 10, number of

In [20]:
def build_PWM_KL(s):
    if isinstance(s, str):
        s = s.split(',')
    n = len(s[0])

    s = np.array([np.array(list(i)) for i in s])

    pfm = np.zeros((len(ALPHABET), n))
    pfm[0]

    #Building PFM
    for row in range(len(pfm)):
        letter = ALPHABET[row]
        for col in range(len(pfm[0])):
            position_array = s[:,col]
            pfm[row][col] = np.sum(np.where(position_array == letter,1,0))


    #Building PPM using Bayesian shrinkage
    ppm = np.zeros((len(ALPHABET), n))
    a = np.sqrt(len(s))

    for row in range(len(ppm)):
        letter = ALPHABET[row]
        for col in range(len(ppm[0])):

            ppm[row][col] = (pfm[row][col] + a * background_freq[letter]) / (len(s) + a)


    #Calculating KL divergence vector from ppm

    kl = np.zeros(n)

    for pos in range(len(kl)):
        for letter in range(len(ALPHABET)):
            q = background_freq[ALPHABET[letter]]
            p = ppm[letter][pos]

            kl[pos] += p * np.log2(p/q)


    #Building PWM
    pwm = np.zeros((len(ALPHABET), n))

    for row in range(len(pwm)):
        letter = ALPHABET[row]
        for col in range(len(pwm[0])):
            pwm[row][col] = np.log2(ppm[row][col] / background_freq[letter])

    return(pwm, ppm, kl)


In [21]:
global background_freq

ALPHABET = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

PWM_PATH = Path('/home/aalarkin/Work/anchorminer/datasets/PWM')
KL_PATH = Path('/home/aalarkin/Work/anchorminer/datasets/KL')
PPM_PATH = Path('/home/aalarkin/Work/anchorminer/datasets/PPM')

for hla in PWM_data:
    for l in PWM_data[hla]:
        peptides = PWM_data[hla][l]
        pwm, ppm, kl = build_PWM_KL(peptides)

        hla_formatted = hla.replace('_', ':').replace('HLA-', 'HLA-')
    
        pwm_filename = f'PWM-{hla_formatted}_{l}.npy'
        np.save(f'{PWM_PATH}/{pwm_filename}', pwm)

        ppm_filename = f'PPM-{hla_formatted}_{l}.npy'
        np.save(f'{PPM_PATH}/{ppm_filename}', ppm)
        
        kl_filename = f'KL-{hla_formatted}_{l}.npy'
        np.save(f'{KL_PATH}/{kl_filename}', np.array(kl)) 
        


In [30]:
old_supported_alleles = pd.read_csv('../datasets/Anchor_Miner_supportedalleles.csv', sep = '\t', index_col=0)


new_supported_alleles = {i: list(PWM_data[i].keys()) for i in PWM_data}
new_supported_alleles = {'allele': list(new_supported_alleles.keys()), 'len': list(new_supported_alleles.values())}
new_supported_alleles_df = pd.DataFrame(new_supported_alleles)



supported_alleles_df = pd.concat((old_supported_alleles, new_supported_alleles_df))
supported_alleles_df = supported_alleles_df.reset_index(drop=True)
display(supported_alleles_df)
supported_alleles_df.to_csv('../datasets/Anchor_Miner_supportedalleles.csv', sep = '\t')


,allele,len
0,HLA-A01_01,"[9, 10, 11, 12]"
1,HLA-A02_01,"[9, 10, 11, 12]"
2,HLA-A02_02,"[9, 10, 11, 12]"
3,HLA-A02_03,"[9, 10, 11]"
4,HLA-A02_04,"[9, 10, 11, 12]"
...,...,...
152,H_2_Kd,"[8, 9, 10, 11]"
153,H_2_Kk,"[8, 9, 10, 11]"
154,H_2_Kq,"[8, 9, 10, 11]"
155,H_2_Ld,"[8, 9, 10, 11, 12]"
